## Get Data

In [1]:
import os
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
from utils8 import AudioCNN
from utils8.data import AudioDataset2, TransformedSubset, get_dataloader
from utils8.augmentations import *
from torchvision.transforms import v2

# Get Data
path = os.path.join('Data', 'Digits')
classes = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']
dataset = AudioDataset2(path, classes=classes)

# Data Split and Subsets
idx = list(range(len(dataset)))
labels = dataset.labels

train_transforms = v2.Compose([
    AudioPitchShift(n_steps=3.0, sr=8000),
    AudioGaussianNoise(snr_db=torch.tensor([20])),
    AudioReverb(sr=8000, room_scale=0.25)
])
train_val_idx, test_idx = train_test_split(idx, test_size=0.2, stratify=labels, random_state=42)

train_val_set = TransformedSubset(Subset(dataset, train_val_idx), train_transforms)
test_set = Subset(dataset, test_idx)

## Get Optuna Params

In [3]:
import pandas as pd

studies = sorted(os.listdir('optuna_results'))

print('Available studies: ')
for s in studies:
    s = s.replace('(', '_')
    s = s.replace(')', '_')
    parsed_s = s.split('_')
    print(f'{parsed_s[1]}/{parsed_s[2]}/{parsed_s[3]} - {parsed_s[4]}')

last_study_path = os.path.join('optuna_results', studies[-1])
last_study = pd.read_csv(last_study_path)

study = last_study.iloc[0].to_dict()

network_col = ["params_dropout_rate", "params_lr", "params_patience", "params_weight_decay"]
best_params = {}
for col in network_col:
    best_params[col.removeprefix('params_')] = study[col]

best_params

Available studies: 
20/05/2026 - 12:08


{'dropout_rate': 0.0082966517228649,
 'lr': 0.0016068082173041,
 'patience': 5,
 'weight_decay': 0.0001426599145501}

## Training

In [ ]:
import os
from sklearn.model_selection import KFold
from torch.utils.tensorboard import SummaryWriter
from torch import nn,optim
from utils8.dir_managment import clean_dir
from utils8.AudioCNN import AudioCNN
from utils8.training import train_one_fold, get_train_loaders, save_model_dict, save_target_labels


model_name = 'model_3'

log_dir = os.path.join('runs', 'training', model_name)
clean_dir(log_dir)
os.makedirs(log_dir, exist_ok=True)
writer = SummaryWriter(log_dir=log_dir)

save_model_dir = os.path.join('Models', model_name)
clean_dir(save_model_dir)
os.makedirs(save_model_dir, exist_ok=True)

model_params = {'dropout_rate': best_params['dropout_rate'], 'num_classes': 10}
save_model_dict(model_params, save_model_dir)
save_target_labels(classes, save_model_dir)

N_EPOCHS = 35
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(kf.split(train_val_set)):
    print(f'\n ===================== Training Fold {fold}  ===================== \n')
    # Loaders
    train_loader, val_loader = get_train_loaders(train_val_set, train_idx,val_idx , batch_size=32)

    # Model, Optimizer, Criterion
    criterion = nn.CrossEntropyLoss()
    model =  AudioCNN(dropout_rate=best_params['dropout_rate'])
    optimizer = optim.Adam(model.parameters(), lr=best_params['lr'], weight_decay=best_params['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=best_params['patience'])

    # Fold Training
    loss = train_one_fold(fold, model, train_loader, val_loader, optimizer, scheduler, criterion, n_epochs=N_EPOCHS, write_model_dir=save_model_dir, writer=writer)


